In [15]:
from pathlib import Path
from pydub import AudioSegment
from config import config

In [16]:
root = Path("../../..")

In [17]:
SOURCE_DIR = root / "data/audio/en"
OUTPUT_DIR = root / config.RAW_AUDIO_DIR_EN
METADATA_FILE = root / config.METADATA_DIR / "metadata_en.csv"
CLIP_DURATION_MS = 3000 # 3 seconds

In [18]:
SOURCE_DIR.exists(), METADATA_FILE.exists() # must be true

(True, True)

In [19]:
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

In [20]:
import math, pandas as pd

In [21]:
N_SUBSET = math.ceil(config.N_SUBSET / 2)

In [22]:
# if n_subset no. of clips are to be considered, then roughly n_subset/10 no. of audio files should be included. to ensure the quadrant distribution of the audio files remain balanced, these n_subset/10 audio files can be sampled randomly first and then their corresponding features extracted??
num_audios = math.ceil(N_SUBSET / 10)

In [23]:
N_SUBSET, num_audios

(1250, 125)

In [24]:
metadata_df = pd.read_csv(METADATA_FILE)

In [25]:
sampled_df = metadata_df.sample(n=num_audios, random_state=42)

In [26]:
sampled_df = sampled_df.reset_index(drop=True)

In [27]:
sampled_df

,audio_file_stem,lyric_file_stem,quadrant,genres,label
0,MT0009968194,MT0009968194,Q1,Country,0
1,MT0004506984,MT0004506984,Q1,"Adult Contemporary,R&B,Contemporary Pop/Rock,D...",0
2,MT0011358600,MT0011358600,Q4,Blues,2
3,A168,L168,Q4,"Electronic,Ethnic Fusion,Club/Dance,New Age",2
4,MT0009239443,MT0009239443,Q2,"International,Pop/Rock",1
...,...,...,...,...,...
120,MT0013350034,MT0013350034,Q1,"Stage & Screen,Vocal",0
121,MT0001889244,MT0001889244,Q1,"Electronic,Pop/Rock,R&B",0
122,MT0007349422,MT0007349422,Q2,Rap,1
123,MT0001511433,MT0001511433,Q4,"Country,Holiday",2


In [28]:
sampled_df = sampled_df[["audio_file_stem", "lyric_file_stem", "quadrant", "genres"]]

In [29]:
audio_file_list = sampled_df["audio_file_stem"].to_list()

In [30]:
len(audio_file_list)

125

In [31]:
audio_file_list[:5]

['MT0009968194', 'MT0004506984', 'MT0011358600', 'A168', 'MT0009239443']

In [32]:
missing = 0
for audio_filename in audio_file_list:
    audio_file = [file for file in SOURCE_DIR.iterdir() if file.is_file() and file.stem == audio_filename][0]
    try:
        if not audio_file: 
            missing += 1
            continue
        audio = AudioSegment.from_file(audio_file)
        total_length = len(audio)
        n_clips = total_length // CLIP_DURATION_MS
        for i in range(n_clips):
            start = i * CLIP_DURATION_MS
            end = start + CLIP_DURATION_MS
            clip = audio[start:end]
            output_file_name = f"{audio_file.stem}_clip_{i+1}.wav"
            output_file_path = OUTPUT_DIR / output_file_name
            clip.export(output_file_path, format="wav")
    except Exception as e:
        print(e)

In [33]:
missing

0

In [34]:
sampled_df.to_csv(METADATA_FILE, index=False)